In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path(
    "../data/raw/paysim/PS_20174392719_1491204439457_log.csv"
)

print("Dataset exists:", DATA_PATH.exists())
print("Dataset path:", DATA_PATH.resolve())

Dataset exists: True
Dataset path: D:\AI-Projects\RiskPilot-AI\data\raw\paysim\PS_20174392719_1491204439457_log.csv


# RiskPilot AI — Feature Engineering

## Decision-time feature policy

RiskPilot makes a fraud-risk decision before the transaction is completed.

### Features used

- step
- type
- amount
- oldbalanceOrg
- oldbalanceDest

### Engineered features

- amount_to_origin_balance
- amount_to_destination_balance

### Excluded features

- nameOrig — high-cardinality identifier
- nameDest — high-cardinality identifier
- newbalanceOrig — post-transaction state
- newbalanceDest — post-transaction state
- isFlaggedFraud — existing rule-based fraud flag
- isFraud — target variable

In [3]:
fraud_parts = []
legitimate_parts = []

RANDOM_STATE = 42

for chunk in pd.read_csv(
    DATA_PATH,
    chunksize=200_000
):

    fraud_chunk = chunk[chunk["isFraud"] == 1]

    legitimate_chunk = chunk[chunk["isFraud"] == 0]

    # Sample approximately 1.5% of legitimate transactions
    # to keep the modeling dataset manageable.
    legitimate_sample = legitimate_chunk.sample(
        frac=0.015,
        random_state=RANDOM_STATE
    )

    fraud_parts.append(fraud_chunk)
    legitimate_parts.append(legitimate_sample)

print("Finished processing all chunks.")

Finished processing all chunks.


In [4]:
df_model = pd.concat(
    fraud_parts + legitimate_parts,
    ignore_index=True
)

print("Modeling dataset shape:", df_model.shape)

Modeling dataset shape: (103530, 11)


In [5]:
print(
    df_model["isFraud"].value_counts()
)

print("\nPercentages:")
print(
    df_model["isFraud"]
    .value_counts(normalize=True)
    .mul(100)
)

isFraud
0    95317
1     8213
Name: count, dtype: int64

Percentages:
isFraud
0    92.067034
1     7.932966
Name: proportion, dtype: float64


In [6]:
df_model["amount_to_origin_balance"] = (
    df_model["amount"] /
    (df_model["oldbalanceOrg"] + 1)
)

In [7]:
df_model["amount_to_destination_balance"] = (
    df_model["amount"] /
    (df_model["oldbalanceDest"] + 1)
)

In [8]:
df_model = pd.get_dummies(
    df_model,
    columns=["type"],
    prefix="type",
    dtype=int
)

In [9]:
feature_columns = [
    "step",
    "amount",
    "oldbalanceOrg",
    "oldbalanceDest",
    "amount_to_origin_balance",
    "amount_to_destination_balance",
    "type_CASH_IN",
    "type_CASH_OUT",
    "type_DEBIT",
    "type_PAYMENT",
    "type_TRANSFER"
]

X = df_model[feature_columns]

y = df_model["isFraud"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (103530, 11)
y shape: (103530,)


In [10]:
# Sort transactions chronologically
df_model = df_model.sort_values("step").reset_index(drop=True)

print("Minimum step:", df_model["step"].min())
print("Maximum step:", df_model["step"].max())

Minimum step: 1
Maximum step: 743


In [11]:
# RiskPilot chronological train / validation / test split

TRAIN_END = 520
VALIDATION_END = 630

train_df = df_model[df_model["step"] <= TRAIN_END].copy()

validation_df = df_model[
    (df_model["step"] > TRAIN_END) &
    (df_model["step"] <= VALIDATION_END)
].copy()

test_df = df_model[
    df_model["step"] > VALIDATION_END
].copy()

print("Training set:", train_df.shape)
print("Validation set:", validation_df.shape)
print("Test set:", test_df.shape)

print("\nTraining steps:",
      train_df["step"].min(), "to", train_df["step"].max())

print("Validation steps:",
      validation_df["step"].min(), "to", validation_df["step"].max())

print("Test steps:",
      test_df["step"].min(), "to", test_df["step"].max())

Training set: (96933, 17)
Validation set: (4035, 17)
Test set: (2562, 17)

Training steps: 1 to 520
Validation steps: 521 to 630
Test steps: 631 to 743


In [12]:
# Step 33: Create chronological ML datasets

X_train = train_df[feature_columns]
y_train = train_df["isFraud"]

X_validation = validation_df[feature_columns]
y_validation = validation_df["isFraud"]

X_test = test_df[feature_columns]
y_test = test_df["isFraud"]

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_validation shape:", X_validation.shape)
print("y_validation shape:", y_validation.shape)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

print("\nFraud distribution:")
print("Train:")
print(y_train.value_counts())

print("\nValidation:")
print(y_validation.value_counts())

print("\nTest:")
print(y_test.value_counts())

X_train shape: (96933, 11)
y_train shape: (96933,)
X_validation shape: (4035, 11)
y_validation shape: (4035,)
X_test shape: (2562, 11)
y_test shape: (2562,)

Fraud distribution:
Train:
isFraud
0    91152
1     5781
Name: count, dtype: int64

Validation:
isFraud
0    2867
1    1168
Name: count, dtype: int64

Test:
isFraud
0    1298
1    1264
Name: count, dtype: int64


In [14]:
# Step 34: Train baseline Random Forest fraud model

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

print("Random Forest training completed successfully.")

Random Forest training completed successfully.


In [15]:
# Step 35: Evaluate Random Forest on validation set

validation_probabilities = rf_model.predict_proba(X_validation)[:, 1]

DEFAULT_THRESHOLD = 0.50

validation_predictions = (
    validation_probabilities >= DEFAULT_THRESHOLD
).astype(int)

print("Confusion Matrix:")
print(confusion_matrix(y_validation, validation_predictions))

print("\nClassification Report:")
print(
    classification_report(
        y_validation,
        validation_predictions,
        digits=4
    )
)

print(
    "Precision:",
    round(
        precision_score(
            y_validation,
            validation_predictions,
            zero_division=0
        ),
        4
    )
)

print(
    "Recall:",
    round(
        recall_score(
            y_validation,
            validation_predictions,
            zero_division=0
        ),
        4
    )
)

print(
    "F1 Score:",
    round(
        f1_score(
            y_validation,
            validation_predictions,
            zero_division=0
        ),
        4
    )
)

print(
    "ROC-AUC:",
    round(
        roc_auc_score(
            y_validation,
            validation_probabilities
        ),
        4
    )
)

print(
    "PR-AUC:",
    round(
        average_precision_score(
            y_validation,
            validation_probabilities
        ),
        4
    )
)

Confusion Matrix:
[[2867    0]
 [   0 1168]]

Classification Report:
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000      2867
           1     1.0000    1.0000    1.0000      1168

    accuracy                         1.0000      4035
   macro avg     1.0000    1.0000    1.0000      4035
weighted avg     1.0000    1.0000    1.0000      4035

Precision: 1.0
Recall: 1.0
F1 Score: 1.0
ROC-AUC: 1.0
PR-AUC: 1.0


In [16]:
# Step 36: Inspect Random Forest feature importance

feature_importance = pd.DataFrame({
    "feature": feature_columns,
    "importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

print(feature_importance)

                          feature  importance
0        amount_to_origin_balance    0.462352
1                   oldbalanceOrg    0.130327
2                          amount    0.070135
3   amount_to_destination_balance    0.070127
4                    type_CASH_IN    0.064452
5                    type_PAYMENT    0.063365
6                   type_TRANSFER    0.055663
7                   type_CASH_OUT    0.053335
8                  oldbalanceDest    0.020807
9                            step    0.008989
10                     type_DEBIT    0.000450


In [17]:
# Step 37: Inspect validation risk-score distribution

validation_results = pd.DataFrame({
    "actual": y_validation.values,
    "risk_score": validation_probabilities
})

print("Minimum risk score:",
      validation_results["risk_score"].min())

print("Maximum risk score:",
      validation_results["risk_score"].max())

print("\nRisk score statistics:")
print(validation_results["risk_score"].describe())

print("\nLegitimate transaction scores:")
print(
    validation_results.loc[
        validation_results["actual"] == 0,
        "risk_score"
    ].describe()
)

print("\nFraud transaction scores:")
print(
    validation_results.loc[
        validation_results["actual"] == 1,
        "risk_score"
    ].describe()
)

Minimum risk score: 0.0
Maximum risk score: 0.9999855220517394

Risk score statistics:
count    4035.000000
mean        0.297400
std         0.446788
min         0.000000
25%         0.000000
50%         0.005506
75%         0.990213
max         0.999986
Name: risk_score, dtype: float64

Legitimate transaction scores:
count    2867.000000
mean        0.012816
std         0.032544
min         0.000000
25%         0.000000
50%         0.002917
75%         0.008684
max         0.418988
Name: risk_score, dtype: float64

Fraud transaction scores:
count    1168.000000
mean        0.995946
std         0.008889
min         0.868785
25%         0.995151
50%         0.999126
75%         0.999943
max         0.999986
Name: risk_score, dtype: float64


In [18]:
# Step 38: RiskPilot cost-aware decision engine

def calculate_action_costs(
    risk_probability,
    amount,
    fraud_loss_rate=1.0,
    false_positive_rate=0.03,
    review_cost=5.0
):
    """
    Calculate the expected monetary cost of each possible action.

    APPROVE:
        Expected fraud loss if the transaction is actually fraudulent.

    BLOCK:
        Cost of blocking a legitimate transaction.

    REVIEW:
        Fixed operational cost of manual review.
    """

    fraud_loss = (
        risk_probability *
        amount *
        fraud_loss_rate
    )

    false_positive_loss = (
        (1 - risk_probability) *
        amount *
        false_positive_rate
    )

    manual_review_cost = review_cost

    return {
        "APPROVE": fraud_loss,
        "REVIEW": manual_review_cost,
        "BLOCK": false_positive_loss
    }


def riskpilot_decision(
    risk_probability,
    amount,
    fraud_loss_rate=1.0,
    false_positive_rate=0.03,
    review_cost=5.0
):
    """
    Select the action with the lowest expected cost.
    """

    costs = calculate_action_costs(
        risk_probability=risk_probability,
        amount=amount,
        fraud_loss_rate=fraud_loss_rate,
        false_positive_rate=false_positive_rate,
        review_cost=review_cost
    )

    recommended_action = min(
        costs,
        key=costs.get
    )

    return recommended_action, costs


print("RiskPilot cost-aware decision engine created successfully.")

RiskPilot cost-aware decision engine created successfully.


In [19]:
# Step 39: Test RiskPilot decision engine

examples = [
    {
        "name": "Low-risk transaction",
        "risk": 0.02,
        "amount": 500
    },
    {
        "name": "Medium-risk transaction",
        "risk": 0.35,
        "amount": 5000
    },
    {
        "name": "High-risk transaction",
        "risk": 0.90,
        "amount": 10000
    }
]

for example in examples:

    action, costs = riskpilot_decision(
        risk_probability=example["risk"],
        amount=example["amount"]
    )

    print("\n", example["name"])
    print("Risk probability:", example["risk"])
    print("Transaction amount:", example["amount"])
    print("Action:", action)
    print("Costs:", costs)


 Low-risk transaction
Risk probability: 0.02
Transaction amount: 500
Action: REVIEW
Costs: {'APPROVE': 10.0, 'REVIEW': 5.0, 'BLOCK': 14.7}

 Medium-risk transaction
Risk probability: 0.35
Transaction amount: 5000
Action: REVIEW
Costs: {'APPROVE': 1750.0, 'REVIEW': 5.0, 'BLOCK': 97.5}

 High-risk transaction
Risk probability: 0.9
Transaction amount: 10000
Action: REVIEW
Costs: {'APPROVE': 9000.0, 'REVIEW': 5.0, 'BLOCK': 29.999999999999993}


In [20]:
# Step 40: Improved RiskPilot cost model

def calculate_action_costs(
    risk_probability,
    amount,
    fraud_loss_rate=1.0,
    false_positive_rate=0.03,
    review_fixed_cost=5.0,
    review_percentage=0.005
):
    """
    Calculate expected cost for each possible merchant action.

    These parameters are configurable assumptions for the prototype.
    They do not represent Razorpay's actual internal costs.
    """

    # Cost if we APPROVE and the transaction turns out to be fraud
    approve_cost = (
        risk_probability *
        amount *
        fraud_loss_rate
    )

    # Cost if we BLOCK a transaction that would have been legitimate
    block_cost = (
        (1 - risk_probability) *
        amount *
        false_positive_rate
    )

    # Manual review has both a fixed operational cost
    # and a small transaction-value-dependent component
    review_cost = (
        review_fixed_cost +
        amount * review_percentage
    )

    return {
        "APPROVE": approve_cost,
        "REVIEW": review_cost,
        "BLOCK": block_cost
    }


def riskpilot_decision(
    risk_probability,
    amount,
    fraud_loss_rate=1.0,
    false_positive_rate=0.03,
    review_fixed_cost=5.0,
    review_percentage=0.005
):
    """
    Select the action with the lowest expected cost.
    """

    costs = calculate_action_costs(
        risk_probability=risk_probability,
        amount=amount,
        fraud_loss_rate=fraud_loss_rate,
        false_positive_rate=false_positive_rate,
        review_fixed_cost=review_fixed_cost,
        review_percentage=review_percentage
    )

    recommended_action = min(
        costs,
        key=costs.get
    )

    return recommended_action, costs


print("Improved RiskPilot cost-aware decision engine created successfully.")

Improved RiskPilot cost-aware decision engine created successfully.


In [21]:
# Step 41: Test improved RiskPilot decision engine

examples = [
    {
        "name": "Very low-risk transaction",
        "risk": 0.005,
        "amount": 500
    },
    {
        "name": "Medium-risk transaction",
        "risk": 0.35,
        "amount": 5000
    },
    {
        "name": "High-risk transaction",
        "risk": 0.90,
        "amount": 10000
    }
]

for example in examples:

    action, costs = riskpilot_decision(
        risk_probability=example["risk"],
        amount=example["amount"]
    )

    print("\n" + example["name"])
    print("Risk probability:", example["risk"])
    print("Transaction amount: ₹", example["amount"])
    print("Action:", action)

    print("Expected costs:")
    for action_name, cost in costs.items():
        print(
            f"  {action_name}: ₹{cost:.2f}"
        )


Very low-risk transaction
Risk probability: 0.005
Transaction amount: ₹ 500
Action: APPROVE
Expected costs:
  APPROVE: ₹2.50
  REVIEW: ₹7.50
  BLOCK: ₹14.92

Medium-risk transaction
Risk probability: 0.35
Transaction amount: ₹ 5000
Action: REVIEW
Expected costs:
  APPROVE: ₹1750.00
  REVIEW: ₹30.00
  BLOCK: ₹97.50

High-risk transaction
Risk probability: 0.9
Transaction amount: ₹ 10000
Action: BLOCK
Expected costs:
  APPROVE: ₹9000.00
  REVIEW: ₹55.00
  BLOCK: ₹30.00


In [22]:
# Step 42: Create merchant-specific cost profiles

merchant_profiles = {

    "SMALL_MERCHANT": {
        "fraud_loss_rate": 1.0,
        "false_positive_rate": 0.02,
        "review_fixed_cost": 5.0,
        "review_percentage": 0.005
    },

    "E_COMMERCE": {
        "fraud_loss_rate": 1.0,
        "false_positive_rate": 0.08,
        "review_fixed_cost": 8.0,
        "review_percentage": 0.003
    },

    "HIGH_VALUE_MERCHANT": {
        "fraud_loss_rate": 1.0,
        "false_positive_rate": 0.15,
        "review_fixed_cost": 10.0,
        "review_percentage": 0.002
    }
}

print("Merchant profiles created:")
for merchant, profile in merchant_profiles.items():

    print("\n", merchant)

    for parameter, value in profile.items():
        print(f"  {parameter}: {value}")

Merchant profiles created:

 SMALL_MERCHANT
  fraud_loss_rate: 1.0
  false_positive_rate: 0.02
  review_fixed_cost: 5.0
  review_percentage: 0.005

 E_COMMERCE
  fraud_loss_rate: 1.0
  false_positive_rate: 0.08
  review_fixed_cost: 8.0
  review_percentage: 0.003

 HIGH_VALUE_MERCHANT
  fraud_loss_rate: 1.0
  false_positive_rate: 0.15
  review_fixed_cost: 10.0
  review_percentage: 0.002


In [23]:
# Step 43: Merchant-aware RiskPilot decision

def merchant_riskpilot_decision(
    risk_probability,
    amount,
    merchant_profile
):

    action, costs = riskpilot_decision(
        risk_probability=risk_probability,
        amount=amount,
        fraud_loss_rate=merchant_profile["fraud_loss_rate"],
        false_positive_rate=merchant_profile["false_positive_rate"],
        review_fixed_cost=merchant_profile["review_fixed_cost"],
        review_percentage=merchant_profile["review_percentage"]
    )

    return action, costs

In [24]:
# Step 44: Same transaction, different merchants

risk = 0.35
amount = 5000

print("Transaction")
print("Risk probability:", risk)
print("Amount: ₹", amount)

for merchant, profile in merchant_profiles.items():

    action, costs = merchant_riskpilot_decision(
        risk_probability=risk,
        amount=amount,
        merchant_profile=profile
    )

    print("\nMerchant:", merchant)
    print("Recommended action:", action)

    print("Expected costs:")

    for action_name, cost in costs.items():
        print(
            f"  {action_name}: ₹{cost:.2f}"
        )

Transaction
Risk probability: 0.35
Amount: ₹ 5000

Merchant: SMALL_MERCHANT
Recommended action: REVIEW
Expected costs:
  APPROVE: ₹1750.00
  REVIEW: ₹30.00
  BLOCK: ₹65.00

Merchant: E_COMMERCE
Recommended action: REVIEW
Expected costs:
  APPROVE: ₹1750.00
  REVIEW: ₹23.00
  BLOCK: ₹260.00

Merchant: HIGH_VALUE_MERCHANT
Recommended action: REVIEW
Expected costs:
  APPROVE: ₹1750.00
  REVIEW: ₹20.00
  BLOCK: ₹487.50


In [25]:
# Step 45A: Calculate review priority

def calculate_review_priority(
    risk_probability,
    amount,
    merchant_profile
):

    _, costs = merchant_riskpilot_decision(
        risk_probability=risk_probability,
        amount=amount,
        merchant_profile=merchant_profile
    )

    approve_cost = costs["APPROVE"]
    review_cost = costs["REVIEW"]

    # Economic value of sending this transaction
    # to human review instead of automatically approving it.
    review_priority = approve_cost - review_cost

    return review_priority

In [26]:
# Step 45B: Test review priority

merchant = merchant_profiles["E_COMMERCE"]

test_transactions = [
    {"risk": 0.05, "amount": 500},
    {"risk": 0.20, "amount": 1000},
    {"risk": 0.35, "amount": 5000},
    {"risk": 0.70, "amount": 8000},
    {"risk": 0.90, "amount": 10000}
]

for transaction in test_transactions:

    priority = calculate_review_priority(
        risk_probability=transaction["risk"],
        amount=transaction["amount"],
        merchant_profile=merchant
    )

    print(
        f"Risk: {transaction['risk']:.2f} | "
        f"Amount: ₹{transaction['amount']:,.2f} | "
        f"Review Priority: ₹{priority:,.2f}"
    )

Risk: 0.05 | Amount: ₹500.00 | Review Priority: ₹15.50
Risk: 0.20 | Amount: ₹1,000.00 | Review Priority: ₹189.00
Risk: 0.35 | Amount: ₹5,000.00 | Review Priority: ₹1,727.00
Risk: 0.70 | Amount: ₹8,000.00 | Review Priority: ₹5,568.00
Risk: 0.90 | Amount: ₹10,000.00 | Review Priority: ₹8,962.00


In [27]:
# Step 45C: Build a merchant review queue

def build_review_queue(
    transactions,
    merchant_profile,
    review_capacity
):

    queue = transactions.copy()

    queue["review_priority"] = queue.apply(
        lambda row: calculate_review_priority(
            risk_probability=row["risk_probability"],
            amount=row["amount"],
            merchant_profile=merchant_profile
        ),
        axis=1
    )

    # Highest economic value first
    queue = queue.sort_values(
        "review_priority",
        ascending=False
    )

    # Only send the highest-priority transactions
    # to human reviewers.
    review_queue = queue.head(review_capacity).copy()

    return review_queue

In [28]:
# Step 45D: Create simulated transaction risk scores

np.random.seed(42)

simulation_size = 1000

simulation_transactions = pd.DataFrame({

    "transaction_id": range(1, simulation_size + 1),

    "risk_probability": np.random.beta(
        a=1,
        b=8,
        size=simulation_size
    ),

    "amount": np.random.lognormal(
        mean=8,
        sigma=1.5,
        size=simulation_size
    )
})

simulation_transactions["amount"] = (
    simulation_transactions["amount"]
    .round(2)
)

print("Simulated transactions:")
print(simulation_transactions.head())

print("\nTotal transactions:",
      len(simulation_transactions))

Simulated transactions:
   transaction_id  risk_probability    amount
0               1          0.086089    897.80
1               2          0.006923   2710.02
2               3          0.129682  19075.45
3               4          0.054424   1501.24
4               5          0.052744   2795.50

Total transactions: 1000


In [29]:
# Step 45E: Generate limited human-review queue

merchant = merchant_profiles["E_COMMERCE"]

REVIEW_CAPACITY = 50

review_queue = build_review_queue(
    simulation_transactions,
    merchant_profile=merchant,
    review_capacity=REVIEW_CAPACITY
)

print("Total transactions:",
      len(simulation_transactions))

print("Human review capacity:",
      REVIEW_CAPACITY)

print("Transactions selected for review:",
      len(review_queue))

print("\nTop review candidates:")

print(
    review_queue[
        [
            "transaction_id",
            "risk_probability",
            "amount",
            "review_priority"
        ]
    ].head(10)
)

Total transactions: 1000
Human review capacity: 50
Transactions selected for review: 50

Top review candidates:
     transaction_id  risk_probability      amount  review_priority
307             308          0.076639  1076640.99     79274.471567
213             214          0.277257   169320.81     46429.453537
846             847          0.279423    90097.90     24897.107534
939             940          0.297912    66294.32     19543.017279
824             825          0.323373    59120.32     18932.574581
401             402          0.140461   111025.48     15253.687034
792             793          0.185575    76694.41     13994.494937
724             725          0.189023    68123.03     12664.464451
946             947          0.185573    69131.46     12613.507007
46               47          0.298088    38175.00     11256.974281


In [30]:
# Step 46: Generate real ML risk scores

test_results = test_df.copy()

test_results["risk_probability"] = (
    rf_model.predict_proba(X_test)[:, 1]
)

print("Real ML risk scores generated.")

print("\nTest transactions:")
print(len(test_results))

print("\nRisk score statistics:")
print(
    test_results["risk_probability"].describe()
)

Real ML risk scores generated.

Test transactions:
2562

Risk score statistics:
count    2562.000000
mean        0.497709
std         0.492019
min         0.000000
25%         0.000140
50%         0.158694
75%         0.998810
max         0.999986
Name: risk_probability, dtype: float64


In [32]:
# Step 47: Add transaction identifiers

test_results["transaction_id"] = range(
    1,
    len(test_results) + 1
)

# Recover transaction type from one-hot encoded columns
type_columns = [
    "type_CASH_IN",
    "type_CASH_OUT",
    "type_DEBIT",
    "type_PAYMENT",
    "type_TRANSFER"
]

test_results["transaction_type"] = (
    test_results[type_columns]
    .idxmax(axis=1)
    .str.replace("type_", "", regex=False)
)

print(
    test_results[
        [
            "transaction_id",
            "step",
            "transaction_type",
            "amount",
            "risk_probability",
            "isFraud"
        ]
    ].head(10)
)

        transaction_id  step transaction_type       amount  risk_probability  \
100968               1   631         TRANSFER       341.30          0.992637   
100969               2   631         TRANSFER    696575.46          0.999945   
100970               3   631         CASH_OUT    696575.46          0.996667   
100971               4   631         TRANSFER    952234.12          0.998449   
100972               5   631         TRANSFER   3120690.98          0.999954   
100973               6   631         CASH_OUT  10000000.00          0.999977   
100974               7   631         CASH_OUT    952234.12          0.996667   
100975               8   631         CASH_OUT   2391612.36          0.999883   
100976               9   631         TRANSFER   2391612.36          0.999952   
100977              10   631         CASH_OUT       341.30          0.964749   

        isFraud  
100968        1  
100969        1  
100970        1  
100971        1  
100972        1  
100973     

In [34]:
print(test_results.columns.tolist())

['step', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud', 'amount_to_origin_balance', 'amount_to_destination_balance', 'type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER', 'risk_probability', 'transaction_id', 'transaction_type']


In [35]:
# Step 48: Build review queue using real ML predictions

merchant = merchant_profiles["E_COMMERCE"]

REVIEW_CAPACITY = 50

real_review_queue = build_review_queue(
    test_results[
        [
            "transaction_id",
            "risk_probability",
            "amount"
        ]
    ],
    merchant_profile=merchant,
    review_capacity=REVIEW_CAPACITY
)

print("Total test transactions:",
      len(test_results))

print("Human review capacity:",
      REVIEW_CAPACITY)

print("Selected for human review:",
      len(real_review_queue))

print("\nTop review candidates:")

print(
    real_review_queue[
        [
            "transaction_id",
            "risk_probability",
            "amount",
            "review_priority"
        ]
    ].head(10)
)

Total test transactions: 2562
Human review capacity: 50
Selected for human review: 50

Top review candidates:
        transaction_id  risk_probability      amount  review_priority
101840             873          0.999983  10000000.0     9.969822e+06
103070            2103          0.999983  10000000.0     9.969822e+06
101279             312          0.999982  10000000.0     9.969816e+06
101280             313          0.999982  10000000.0     9.969816e+06
102655            1688          0.999982  10000000.0     9.969816e+06
103120            2153          0.999982  10000000.0     9.969816e+06
103486            2519          0.999980  10000000.0     9.969791e+06
101817             850          0.999977  10000000.0     9.969766e+06
101712             745          0.999977  10000000.0     9.969766e+06
100973               6          0.999977  10000000.0     9.969766e+06


In [36]:
# Step 49: Analyze economic exposure of the review queue

total_test_amount = test_results["amount"].sum()

review_amount = real_review_queue["amount"].sum()

review_transaction_percentage = (
    len(real_review_queue) /
    len(test_results)
) * 100

review_value_percentage = (
    review_amount /
    total_test_amount
) * 100

print("=== RiskPilot Review Queue Analysis ===")

print(
    f"Total test transactions: "
    f"{len(test_results):,}"
)

print(
    f"Human review capacity: "
    f"{len(real_review_queue):,}"
)

print(
    f"Review capacity used: "
    f"{review_transaction_percentage:.2f}%"
)

print(
    f"\nTotal transaction value: "
    f"₹{total_test_amount:,.2f}"
)

print(
    f"Transaction value covered by review queue: "
    f"₹{review_amount:,.2f}"
)

print(
    f"Transaction value concentrated in review queue: "
    f"{review_value_percentage:.2f}%"
)

=== RiskPilot Review Queue Analysis ===
Total test transactions: 2,562
Human review capacity: 50
Review capacity used: 1.95%

Total transaction value: ₹2,371,315,749.50
Transaction value covered by review queue: ₹499,993,773.28
Transaction value concentrated in review queue: 21.09%


In [37]:
# Step 50A: Calculate economic review priority
# for every test transaction

test_results["review_priority"] = test_results.apply(
    lambda row: calculate_review_priority(
        risk_probability=row["risk_probability"],
        amount=row["amount"],
        merchant_profile=merchant_profiles["E_COMMERCE"]
    ),
    axis=1
)

print("Review priorities calculated.")

print(
    test_results[
        [
            "transaction_id",
            "risk_probability",
            "amount",
            "review_priority"
        ]
    ].head(10)
)

Review priorities calculated.
        transaction_id  risk_probability       amount  review_priority
100968               1          0.992637       341.30     3.297632e+02
100969               2          0.999945    696575.46     6.944395e+05
100970               3          0.996667    696575.46     6.921561e+05
100971               4          0.998449    952234.12     9.478928e+05
100972               5          0.999954   3120690.98     3.111177e+06
100973               6          0.999977  10000000.00     9.969766e+06
100974               7          0.996667    952234.12     9.461960e+05
100975               8          0.999883   2391612.36     2.384149e+06
100976               9          0.999952   2391612.36     2.384315e+06
100977              10          0.964749       341.30     3.202449e+02


In [38]:
# Step 50B: Compare risk-only vs economic-priority review

REVIEW_CAPACITY = 50

# Strategy A:
# Review the 50 highest-risk transactions
risk_only_queue = test_results.nlargest(
    REVIEW_CAPACITY,
    "risk_probability"
).copy()

# Strategy B:
# Review the 50 transactions with the
# highest economic review priority
economic_queue = test_results.nlargest(
    REVIEW_CAPACITY,
    "review_priority"
).copy()

print("=== Strategy Comparison ===")

print("\nStrategy A: Highest Risk")
print(
    "Transactions reviewed:",
    len(risk_only_queue)
)

print(
    "Total transaction value:",
    f"₹{risk_only_queue['amount'].sum():,.2f}"
)

print(
    "Average risk:",
    f"{risk_only_queue['risk_probability'].mean():.4f}"
)

print("\nStrategy B: RiskPilot Economic Priority")
print(
    "Transactions reviewed:",
    len(economic_queue)
)

print(
    "Total transaction value:",
    f"₹{economic_queue['amount'].sum():,.2f}"
)

print(
    "Average risk:",
    f"{economic_queue['risk_probability'].mean():.4f}"
)

=== Strategy Comparison ===

Strategy A: Highest Risk
Transactions reviewed: 50
Total transaction value: ₹291,417,634.42
Average risk: 1.0000

Strategy B: RiskPilot Economic Priority
Transactions reviewed: 50
Total transaction value: ₹499,993,773.28
Average risk: 0.9998


In [40]:
# Step 50C: Compare fraud captured by both strategies

risk_only_fraud = risk_only_queue["isFraud"].sum()

economic_fraud = economic_queue["isFraud"].sum()

total_test_fraud = test_results["isFraud"].sum()

risk_only_capture_rate = (
    risk_only_fraud / total_test_fraud * 100
    if total_test_fraud > 0
    else 0
)

economic_capture_rate = (
    economic_fraud / total_test_fraud * 100
    if total_test_fraud > 0
    else 0
)

print("=== Fraud Capture Comparison ===")

print("\nTotal fraud transactions in test set:")
print(total_test_fraud)

print("\nStrategy A: Highest Risk")
print("Fraud transactions captured:", risk_only_fraud)
print(
    "Fraud capture rate:",
    f"{risk_only_capture_rate:.2f}%"
)

print("\nStrategy B: RiskPilot Economic Priority")
print("Fraud transactions captured:", economic_fraud)
print(
    "Fraud capture rate:",
    f"{economic_capture_rate:.2f}%"
)

=== Fraud Capture Comparison ===

Total fraud transactions in test set:
1264

Strategy A: Highest Risk
Fraud transactions captured: 50
Fraud capture rate: 3.96%

Strategy B: RiskPilot Economic Priority
Fraud transactions captured: 50
Fraud capture rate: 3.96%
